# App guide: Nicolas

Conflict, Democracy and Arms project. This notebook walks through all 12 tasks of `app/build_nicolas.py`,
in the same order, and explains what each part of the app does, why it is built that way and how it works.
Only Nicolas edits this file.

**Why a notebook for an app?** Streamlit commands like `st.metric` or `st.slider` only work inside the
running app. In the app you see only the finished page, so it is hard to tell which line went wrong.
Most of the work in the app is plain pandas, matplotlib and Plotly, though: find the right rows, pick a value,
draw a figure. That part you can build and test here, cell by cell, with checkpoints. Then you move it
into the app, where only the display commands are new.

**How every step works**

1. **What you need:** the new idea in a few sentences, with a small example on made-up data. Run it.
2. **Your turn:** the same idea on our data. You write the code in the empty cell.
3. **Hints:** only if you are stuck.
4. **Checkpoint:** a test cell. OK means the step is done.
5. **Move it to the app:** a short table shows which notebook line becomes which app line.
   Then run `python tools/check_build.py` in the terminal.

| Part | What | Status in your app |
| --- | --- | --- |
| 0 | How Streamlit thinks: reruns, the translation table | understanding |
| Tasks 1 to 2 | Top bar, layout | done: redo here, then compare |
| Tasks 3 to 6 | World map, conflict highlight, map click, animation | done: redo here, then compare |
| Tasks 7 to 8 | Main finding chart, source cards | done: redo here, then compare |
| Tasks 9 to 10 | Year card, country card with timelines | open |
| Task 11 | Period buttons, year slider, comparison | open |
| Task 12 | Data and method page | open |
| After task 12 | What comes next | |

Keep the app running while you work: `streamlit run app/build_nicolas.py` in terminal 1.

## Setup

Run this every time you open the notebook. It loads the same tables as the GIVEN block at the top of
the app, under the same names, so code you write here works there unchanged.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt

CLEAN = "../data/clean/"
sys.path.append("../app")          # so Python finds app/panels_conflict.py
from panels_conflict import regime_bands, conflict_bars

panel = pd.read_csv(CLEAN + "panel.csv")
vdem_regime_spans = pd.read_csv(CLEAN + "vdem_regime_spans.csv")
ucdp_spans = pd.read_csv(CLEAN + "ucdp_spans.csv")
sipri_suppliers = pd.read_csv(CLEAN + "sipri_suppliers.csv")
event_time = pd.read_csv(CLEAN + "event_time.csv")
event = pd.read_csv(CLEAN + "event.csv")

names = sorted(panel["country_name"].unique())

pd.set_option("display.max_columns", 50)
print(len(panel), "rows in panel,", len(names), "countries")

---
# Part 0: How Streamlit thinks

Two ideas explain almost everything in the app. Read this part once before task 1.

## 0.1 The whole file runs again after every click

#### What you need

In Jupyter you run one cell at a time and your variables stay in memory.
Streamlit works differently: it runs the **whole** `.py` file from the first line to the last, and draws
the page while it goes. When the user moves a slider or clicks a button, Streamlit runs the whole file
**again**, from the top.

A widget gives back its current value:
`year = st.slider("Year", 1950, 2025, 2000)` puts the number the slider shows right now into `year`.
The first time it is 2000. After the user drags it to 1960, the file runs again and this time `year` is 1960.
So every line below the slider simply works with the current value. You never write "when the slider
moves, do this". You write normal top-to-bottom code, and the rerun does the rest.

The cell below imitates this with a function. Calling the function stands for "Streamlit runs the file".

In [ ]:
# Example: run this cell and look at the output
def run_app(slider_value):
    # this function stands for the whole build_nicolas.py file
    year = slider_value                # in the app: year = st.slider("Year", 1950, 2025, 2000)
    print("--- the file runs from the top ---")
    print("the map shows the year", year)


run_app(2000)    # the page opens: the slider shows its start value
run_app(1960)    # the user drags the slider to 1960: everything runs again

## 0.2 From Jupyter to Streamlit: the translation table

Most of the app is code you already know from the notebooks. Only the last step, showing something, changes:

| In Jupyter | In the app | Note |
| --- | --- | --- |
| `print(x)` | `st.write(x)` | |
| `display(table)` or just `table` | `st.dataframe(table, hide_index=True)` | an interactive table |
| `plt.show()` | `st.pyplot(fig)` then `plt.close(fig)` | |
| `fig.show()` (Plotly) | `st.plotly_chart(fig)` | |
| a variable you set by hand | a widget: `st.slider`, `st.selectbox`, `st.segmented_control` | the user sets it |
| a big number you print | `st.metric("label", value)` | |

Two lines from the GIVEN block and why they are there:
- `@st.cache_data` above `load()`: the file reruns after every click. Without the cache, every click would
  read all CSV files again and the app would feel slow. With it, each file is read once and kept.
- `plt.close(fig)`: every rerun creates new figures. matplotlib keeps every figure in memory until it is
  closed, so without this line the app gets slower the longer someone uses it.

**How the tasks below work.** For each task you build the data part here and test it. Then a short
"Move it to the app" table shows which lines change. Tasks 1 to 8 already pass `check_build.py` in your app:
for those, do the steps here anyway, then open the task in `build_nicolas.py` and compare it with what you
wrote. If you can explain every line there, the task is really done. Steps marked **Extra** are optional
improvements for the app.

---
# Task 1: The top bar

**What you build:** one line at the top: the app name, three page buttons, a country search.

**Why:** the reader needs two choices on every page: which page, which country. Putting both in one bar
at the top keeps them in the same place everywhere, so nobody has to look for them.

**How:** `st.segmented_control` returns the text of the clicked button, `st.selectbox` the chosen country.
Further down, `if page == ...` decides which page is drawn.

## 1.1 The country list

#### What you need

`panel["country_name"].unique()` gives every name once. `sorted(...)` puts them in alphabetical order, so
the dropdown is easy to scan. The GIVEN block stores this list as `names`.

#### Your turn

**Step 1.** Make the list yourself and call it `my_names`. Print how many countries it has and the first three.

In [ ]:
# your code here

my_names = sorted(panel["country_name"].unique())

print(my_names)


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert my_names == names
assert len(my_names) == 182
assert my_names[0] == "Afghanistan", "sorted() puts the names in alphabetical order"
print("OK")

## 1.2 Which page is drawn

#### What you need

`page = st.segmented_control(...)` gives back the text of the clicked button. One detail: if the user
clicks the button that is already active, it switches off and gives back `None`. That is why the app has
`if page is None: page = "Overview"`. Then `if / elif / else` decides what is drawn.

#### Your turn

**Step 1.** Write a function `run_page(page)` that imitates this. It returns (not prints) a text:
- `"map"` for `"Overview"` and for `None`
- `"country card"` for `"Country timeline"`
- `"method"` for anything else

Then call it with all four inputs and print the results.

In [ ]:
# your code here

def run_page(page):
    if page is None:
        page = "Overview"
    if page == "Overview":
        return "map"
    elif page == "Country timeline":
        return "country card"
    else:
        return "method"
        


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Start with the same two lines as in your app: `if page is None:` and below it `page = "Overview"`.
Then an `if / elif / else` with one `return` in each branch.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert run_page("Overview") == "map"
assert run_page(None) == "map", "None should count as Overview"
assert run_page("Country timeline") == "country card"
assert run_page("Data and method") == "method"
print("OK")

## 1.3 Why the widgets have a key

#### What you need

Because the file starts again from the top after every click, normal variables are forgotten each time.
`st.session_state` is a dictionary that Streamlit keeps between reruns. It is the app's memory.

A widget with `key="country_box"` stores its value in `st.session_state["country_box"]`. And if you put a
value into `st.session_state["country_box"]` **before** the widget is drawn, the widget shows that value.
That is how other parts of the app can change the dropdown: the map click in task 5 uses it.

The GIVEN block above the top bar sets start values with the pattern "only if it is not there yet":

In [ ]:
# Example: a plain dictionary plays the role of st.session_state
state = {}

for run in [1, 2, 3]:                     # three reruns of the file
    if "country_box" not in state:        # true only in the very first run
        state["country_box"] = "Ukraine"
    print("run", run, "->", state)

**Step 1.** Answer in a comment: what would happen if the line were just `state["country_box"] = "Ukraine"`,
without the `if`, and the user had picked Colombia in the dropdown?

In [ ]:
# your code here

#that would result in colombia being overwritten to = Ukraine, with the if it just makes it the default for like the first opening of the script


## Move it to the app: Task 1

| In this notebook | In the app |
| --- | --- |
| `page` given by hand | `page = st.segmented_control("Page", [...], label_visibility="collapsed", key="page_nav")` |
| `my_names` | `country = st.selectbox("Search a country", names, label_visibility="collapsed", key="country_box")` |
| `if page is None: page = "Overview"` | the same line |
| (nothing) | `st.columns([2, 3, 2], vertical_alignment="center")` and one `with` per column |

Compare with the TASK 1 block in your app.

---
# Task 2: Research question and layout

**What you build:** a small grey label, the research question as a heading, one line under it, and two
columns for the map (left, wider) and the main finding (right).

**Why:** the first thing a visitor reads should be the question the app answers. The two columns put the
data (map) and the answer (finding) side by side.

**How:** `st.caption`, `st.header`, `st.write` for text, `st.columns([3, 2])` for the layout.

## 2.1 Indentation decides where things appear

#### What you need

`st.columns`, `st.container` and `st.expander` are places on the page. Everything you write indented under
`with <place>:` is drawn inside that place. Indentation in a Streamlit file is layout: one space too few and an
element lands outside its box.

| Command | What it makes |
| --- | --- |
| `left, right = st.columns(2)` | two columns side by side, same width |
| `st.columns([3, 2])` | two columns, the left one 3/5 of the width |
| `with st.container(border=True):` | a box with a thin border |
| `with st.expander("title"):` | a box that opens on click |

You can nest them: a container inside a column, an expander inside a container.

#### Your turn

Read this code (do not run it, it needs the app):

```python
left, right = st.columns(2)
with left:
    st.write("A")
    with st.container(border=True):
        st.write("B")
st.write("C")
with right:
    st.write("D")
```

**Step 1.** Where does each letter appear? Choose for each one of: `"left"`, `"left box"`, `"right"`, `"below"`
(below both columns). Store the answers in `where_a`, `where_b`, `where_c`, `where_d`.

In [ ]:
# your code here



where_a = "left"
where_b = "left box"
where_c = "below"
where_d = "right"





<details><summary><b>Hint 1</b> (open only if stuck)</summary>

C is not indented at all, so it belongs to no column. It is drawn after the columns, across the full width.
D comes later in the file than C, but `with right:` still puts it into the right column, which sits next to the left one.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert where_a == "left"
assert where_b == "left box"
assert where_c == "below", "C has no indentation: it is not in a column"
assert where_d == "right", "the place is decided by `with right:`, not by the order in the file"
print("OK")

**Step 2.** In your app, everything of the Overview is inside `if page == "Overview":`, so it starts with
4 spaces. The map is inside `with col_map:` and `with st.container(border=True):`. How many spaces are in front
of the slider line? Store the number in `spaces_slider`.

In [ ]:
# your code here

spaces_slider = 12



**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert spaces_slider == 12, "4 for the if, 4 for with col_map, 4 for with st.container"
print("OK")

## Move it to the app: Task 2

Compare with the TASK 2 block in your app. There is nothing to compute in this task; what counts is that you can
say for every line on the Overview which column and which box it is in.

---
# Task 3: The world map

**What you build:** a year slider and a world map, every country coloured by its regime in that year.

**Why:** the map is the first picture of the whole dataset. Moving the slider shows how regimes changed
over 75 years in a few seconds.

**How:** filter `panel` to the slider year, then `px.choropleth` draws one country per row.
Plotly knows every country by its three-letter code, which is why `iso3` was so important in the cleaning.

## 3.1 One year of the panel

#### Your turn

**Step 1.** Make `map_data`: all rows of `panel` for 1960. Store the number of countries in `n_1960`.

In [ ]:
# your code here

map_data = panel[panel["year"] == 1960]

n_1960 = len(map_data)




**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert n_1960 == 155, "filter with == 1960 (a number, not the text \"1960\")"
print("OK")

## 3.2 The choropleth

#### What you need

`px.choropleth(table, locations="iso3", color="regime_label")` makes a world map: one country per row,
found by the code in `locations`, coloured by the column in `color`.
`color_discrete_map=` says which colour each label gets. `hover_name=` is the bold text when the mouse is over a country.
`fig.show()` displays a Plotly figure in the notebook.

Plotly makes one layer (a "trace") per colour, so the countries are spread over several layers in `fig.data`.

In [ ]:
# Example: run this cell
import plotly.express as px

toy = pd.DataFrame({"iso3": ["CHE", "FRA", "ITA"], "kind": ["a", "b", "b"]})
fig = px.choropleth(toy, locations="iso3", color="kind", color_discrete_map={"a": "red", "b": "blue"})
fig.show()
print(len(fig.data), "layers")

#### Your turn

**Step 1.** Run the cell below. It copies the colours from the GIVEN block of your app.

In [ ]:
regime_colours = {
    "closed autocracy": "#b8322f",
    "electoral autocracy": "#efa08d",
    "electoral democracy": "#9ec5f4",
    "liberal democracy": "#1c5cab",
}

**Step 2.** Make the map for 1960 from `map_data`, coloured by `regime_label`, with the country name on hover
and `regime_colours`. Call it `fig` and show it.

In [ ]:
# your code here


fig = px.choropleth(map_data, locations="iso3", color = "regime_label", color_discrete_map = regime_colours, hover_data="country_name")

fig.show()



**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
n_on_map = 0
for trace in fig.data:
    n_on_map = n_on_map + len(trace.locations)
assert n_on_map == 155, "the map should show every country of map_data"
assert len(fig.data) == 4, "one layer per regime label"
print("OK")

**Extra (improves the app).** Add `hover_data={"democracy_index": True, "iso3": False}` to the choropleth.
Now the hover box also shows the democracy score. Try it here, then in the app.

In [ ]:
# your code here





# your code here


fig = px.choropleth(map_data, locations="iso3", color = "regime_label", color_discrete_map = regime_colours, hover_data={"democracy_index": True, "iso3": False})

fig.show()




## Move it to the app: Task 3

| In this notebook | In the app |
| --- | --- |
| `1960` typed by hand | `map_year = st.slider("Year", 1950, 2025, 2000, key="map_year")`, then `== map_year` |
| `fig.show()` | `style_map(fig)` (GIVEN: projection, colours, legend), then `st.plotly_chart(fig, width="stretch", key="map")` |

---
# Task 4: Conflict highlight

**What you build:** countries with an active conflict in strong colour with a dark outline, all others faded.
A switch turns it on and off.

**Why:** the map now shows two things at once: regime (colour) and conflict (strength and outline).
You see directly whether conflicts happen more in autocracies or democracies in a given year.

**How:** split the year into two tables, make one map from each, fade the first, outline the second, and lay
the layers of the second map on top of the first.

## 4.1 Two tables

**Step 1.** Make `map_data` for 2000, then `calm` (`conflict_active == 0`) and `war` (`conflict_active == 1`).
Store the number of conflict countries in `n_war_2000`.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert n_war_2000 == 29
assert len(calm) + len(war) == len(map_data), "calm and war together are all countries of 2000"
print("OK")

## 4.2 Two maps on top of each other

#### What you need

- `fig.update_traces(marker_opacity=0.35)` changes every layer of `fig`: here, 35% visible.
- `fig.update_traces(marker_line_color="#1c1a17", marker_line_width=1.2, showlegend=False)` gives every
  layer a dark outline and hides it in the legend (the legend already lists the regimes once).
- `for trace in war_fig.data: fig.add_trace(trace)` copies the layers of the second map onto the first.

#### Your turn

**Step 1.** Build `fig` from `calm` (faded) and `war_fig` from `war` (outlined, no legend), copy the layers
of `war_fig` onto `fig`, and show it.

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

The TASK 4 block in your app has exactly these lines. The second map is made from `war`, not from `map_data`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
faded = 0
outlined = 0
for trace in fig.data:
    if trace.marker.opacity == 0.35:
        faded = faded + len(trace.locations)
    if trace.marker.line.width == 1.2:
        outlined = outlined + len(trace.locations)
assert faded == len(calm), "the calm countries should be faded"
assert outlined == 29, "the 29 conflict countries should have an outline"
print("OK")

## Move it to the app: Task 4

| In this notebook | In the app |
| --- | --- |
| always both layers | `show_conflicts = st.toggle("Show active conflicts", value=True)`, then `if show_conflicts:` both layers, `else:` the simple map from task 3 |

---
# Task 5: Map click

**What you build:** clicking a country on the map opens its timeline.

**Why:** the map makes people curious about one country. A click is the shortest way from "what is
that red country?" to its whole history.

**How:** `st.plotly_chart(..., on_select="rerun")` reruns the file after a click and gives back the clicked
countries. The click cannot change the dropdown directly, because the dropdown was already drawn higher up
in the file. So the app leaves a note, `st.session_state["goto"]`, and calls `st.rerun()`. In the next run,
the GIVEN block at the top reads the note, sets the dropdown and the page, and deletes the note.

## 5.1 What a click gives back

#### What you need

After a click, `event.selection.points` is a list. It is empty if nothing was clicked. Each clicked country
is a dictionary, and `"location"` holds its code. A click on Colombia looks like this:

In [ ]:
# Example: run this cell
points = [{"curve_number": 2, "point_index": 17, "location": "COL"}]
print(len(points))
print(points[0]["location"])

#### Your turn

**Step 1.** Write `clicked_name(points)`. It returns the country name for the first clicked code, or `None` if
the list is empty or the code is not in `panel` (a country with no data). Check the length before `points[0]`.

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Three steps: `if len(points) == 0: return None`; filter `panel` by `iso3 == clicked`; if that is empty,
`return None`, otherwise the first `country_name`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert clicked_name([{"location": "COL"}]) == "Colombia"
assert clicked_name([]) is None, "no click: None"
assert clicked_name([{"location": "ATA"}]) is None, "Antarctica is not in panel: None"
print("OK")

## 5.2 The note and the switch

**Step 1.** The cell below sets up the memory as it is right after a user clicked Colombia: the page is still
the Overview, and `goto` holds the note. Run it.

In [ ]:
state = {"country_box": "Ukraine", "page_nav": "Overview", "goto": "Colombia", "map": "old click"}
print(state)

**Step 2.** Copy the GIVEN block "map click switch" from your app (it starts with `if "goto" in st.session_state:`)
and write `state` everywhere it says `st.session_state`. Run it and print `state`.

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

`del state["goto"]` removes the key from the dictionary. The block has five lines under the first `if`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert state == {"country_box": "Colombia", "page_nav": "Country timeline"}, "after the switch only these two keys should be left"
print("OK")

**Step 3.** Answer in two comments, in your own words:
1. Why must this block stand **above** the top bar, before the dropdown is drawn?
2. Why does it also delete `"map"` (the stored click on the map)?

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Question 1: the dropdown reads `country_box` at the moment it is drawn. Streamlit even stops with an error
if you change a widget's value after it was drawn in the same run.

Question 2: the map is a widget with `key="map"`, so it also remembers its last selection.
What would happen in the next rerun if that old selection was still there?

</details>

## Move it to the app: Task 5

| In this notebook | In the app |
| --- | --- |
| `points` typed by hand | `event = st.plotly_chart(fig, ..., on_select="rerun", selection_mode="points", key="map")`, then `points = event.selection.points` |
| `return match["country_name"].iloc[0]` | `st.session_state["goto"] = match["country_name"].iloc[0]` and `st.rerun()` |
| `state` | `st.session_state` |

---
# Task 6: Play through the years

**What you build:** a switch that turns the map into an animation with a play button, one picture per year.

**Why:** the waves of democratisation after 1990 and the changes after the Cold War are easier to see as
a film than year by year with the slider.

**How:** `animation_frame="year"` tells Plotly to make one picture ("frame") per value in the `year` column.
Plotly orders the frames in the order the years first appear in the table, so the table is sorted by year first.

#### Your turn

**Step 1.** Make the animated map from `panel.sort_values("year")` and call it `anim`. Show it and press play.
This takes a few seconds because it holds all 76 years.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert len(anim.frames) == 76, "one frame per year from 1950 to 2025"
assert anim.frames[0].name == "1950", "sort by year so the film starts in 1950"
print("OK")

**Step 2.** Answer in a comment: why is the animated map a separate branch (`if animate: ... else: ...`) in the
app, and not just added to the normal map?

In [ ]:
# your code here


## Move it to the app: Task 6

| In this notebook | In the app |
| --- | --- |
| always animated | `animate = st.toggle("Play through the years", value=False)`, then `if animate:` this map, `else:` everything from tasks 3 to 5, indented 4 more spaces |
| `anim.show()` | `style_map(fig)`, `st.plotly_chart(fig, width="stretch")` |

---
# Task 7: The main finding chart

**What you build:** two lines (high and low arms importers) showing how democracy changes around the start
of a conflict, a dashed line at the start, and a fold-out table with the numbers.

**Why:** this chart is the answer to your research question, and it carries the most weight in the grade.
It sits next to the map so nobody has to search for it.

**How:** `dem_change` is the change in the democracy index compared with the year before the conflict started
(`t = -1`, which is why it is 0 there). Two filtered tables, two `ax.plot` calls on one `ax`.

#### Your turn

**Step 1.** Make `high` and `low` from `event_time` (the texts are exactly `"high importer"` and `"low importer"`).
Draw both lines on one `ax`, a dashed line at `t = 0`, axis labels and a legend. Keep the figure in `fig2` and show it.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert len(high) == 16 and len(low) == 16, "16 years (t = -5 to 10) per group"
ax_lines = fig2.axes[0].get_lines()
assert len(ax_lines) == 3, "two data lines and the dashed line at 0"
assert fig2.axes[0].get_xlabel() != "" and fig2.axes[0].get_ylabel() != ""
print("OK")

**Step 2.** Take `dem_change` at `t = 5` for both groups and store it in `high_5` and `low_5`. Then write one
sentence in a comment: what does the chart say five years after a conflict starts?

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Filter `high` by `t == 5`, take the column `dem_change` and `.iloc[0]` to get one number out of the one-row result.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert round(high_5, 3) == 0.021
assert round(low_5, 3) == 0.007
print("OK")

**Extra (improves the app).** The averages at the end of the window rest on fewer onsets. Under the chart,
show how many: `st.caption("Based on " + str(n) + " conflict onsets at t = 10")`. Compute `n_at_10` here as the
sum of `n_countries` at `t = 10` over both groups.

In [ ]:
# your code here


## Move it to the app: Task 7

| In this notebook | In the app |
| --- | --- |
| `plt.show()` | `st.pyplot(fig2)` then `plt.close(fig2)` |
| printing `event_time` | `with st.expander("Show the numbers"): st.dataframe(event_time, hide_index=True)` |
| (nothing) | everything inside `with col_find:` and `with st.container(border=True):`, with `st.caption("MAIN FINDING")` and `st.subheader(...)` on top |

---
# Task 8: The source cards

**What you build:** three boxes in a row, one per dataset: name in bold, one sentence in grey.

**Why:** every chart on the page rests on these three sources. Naming them with version numbers shows the
reader where the numbers come from, and the course grades data sources.

**How:** `st.columns(3)`, and in each column a container with `st.markdown("**name**")` and `st.caption("sentence")`.

#### What you need

When you write the same block three times with different text, a list and a loop keep it short, and a fourth
source would cost one line. `zip(a, b)` walks through two lists side by side.

In [ ]:
# Example: run this cell
cities = [("Zurich", "Largest city"), ("Bern", "Capital"), ("Geneva", "UN seat")]
for name, note in cities:
    print("**" + name + "**", "-", note)

#### Your turn

**Step 1.** Make `sources`: a list of three pairs `(name, sentence)`, one for UCDP, V-Dem and SIPRI, with the
versions you used (UCDP/PRIO v26.1, V-Dem v16, SIPRI trade register). Write the sentences yourself:
what does each dataset measure, in one short sentence? Print them with a loop.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert len(sources) == 3
all_names = " ".join([pair[0] for pair in sources])
for word in ["UCDP", "V-Dem", "SIPRI"]:
    assert word in all_names, word + " is missing"
print("OK")

## Move it to the app: Task 8

Your app writes the three boxes one by one, which is fine. **Extra:** with the list from above, the whole block becomes:

```python
for column, (name, sentence) in zip(st.columns(3), sources):
    with column:
        with st.container(border=True):
            st.markdown("**" + name + "**")
            st.caption(sentence)
```

`st.columns(3)` gives three columns, `zip` pairs each column with one source. Only change it if you can explain both lines.

---
# Task 9: The year card

**What you will build:** four numbers in a row for one country in the chosen year:
regime, democracy index, armed conflicts, main arms supplier.

**Why:** the timelines below show the whole history at once. The year card answers the reader's
first question for the year on the slider: what was this country like in that year? It also changes
when the slider moves, which makes the slider feel alive.

**How:** in the app, `year_card(iso3, year)` finds one row of `panel` and one row of `sipri_suppliers`
and shows four `st.metric` boxes. Here you write the same logic step by step and give back the values
for testing.

## 9.1 One row: a filter with two conditions

#### What you need

To get one country in one year, you need two conditions at once. Join them with `&` and put **each
condition in its own brackets**. Without the brackets Python reads the line in the wrong order and stops
with an error.

In [ ]:
# Example: run this cell
scores = pd.DataFrame({"name": ["Ana", "Ana", "Ben"], "year": [2020, 2021, 2021], "points": [12, 15, 7]})
rows = scores[(scores["name"] == "Ana") & (scores["year"] == 2021)]
print(rows)
print(len(rows), "row")

#### Your turn

**Step 1.** Make `rows`: the rows of `panel` for Ukraine (`"UKR"`) in 2014. Print it.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert len(rows) == 1, "there is exactly one row per country and year"
assert rows["iso3"].iloc[0] == "UKR" and rows["year"].iloc[0] == 2014
print("OK")

## 9.2 From one-row table to single values

#### What you need

`rows` is still a table, even with one row. `rows.iloc[0]` takes out the first row. Then
`row["column"]` gives one value, the way a dictionary does. `round(x, 2)` keeps two decimals.

In [ ]:
# Example: run this cell
row = scores[scores["name"] == "Ben"].iloc[0]
print(row)
print(row["points"])

**Step 1.** Take the row out of `rows` and call it `row`. Store the regime label in `regime` and the democracy
index rounded to 2 decimals in `dem`. Print both.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert regime == "electoral autocracy"
assert dem == 0.42
print("OK")

## 9.3 The conflict text

#### What you need

The third box shows a short text built from two columns:
- `n_conflicts == 0`: `"None"`
- otherwise, if `max_intensity == 2` (at least 1,000 battle deaths): the number and `" · war"`, for example `"4 · war"`
- otherwise: the number and `" · minor"`, for example `"1 · minor"`

`n_conflicts` is a number, so you need `str(...)` before you can join it with text using `+`.
The dot in the middle is a special character: copy `" · war"` from here.

#### Your turn

**Step 1.** Write a function `conflict_text(row)` that receives one row and returns the text.
Test it on Ukraine 2014 (`row` from above), Ukraine 2000 and Algeria (`"DZA"`) 2000.

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Same shape as `run_page` in task 1: `if / elif / else`, with a `return` in each branch.
`str(row["n_conflicts"]) + " · war"` builds the text.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
def one_row(iso3, year):
    return panel[(panel["iso3"] == iso3) & (panel["year"] == year)].iloc[0]

assert conflict_text(one_row("UKR", 2014)) == "4 · war"
assert conflict_text(one_row("UKR", 2000)) == "None"
assert conflict_text(one_row("DZA", 2000)) == "1 · minor"
print("OK")

## 9.4 When there is no row

#### What you need

Ukraine has no V-Dem row before independence. For 1980 the filter gives an empty table, and
`.iloc[0]` on an empty table stops with an `IndexError`. In the app that would be a red error box the
moment someone drags the slider to 1980. So the app checks `len(rows) == 0` **first**, and only takes
`.iloc[0]` when there is something to take.

#### Your turn

**Step 1.** Make `rows_1980` for Ukraine in 1980 and print its length.

**Step 2.** In the next cell, run `rows_1980.iloc[0]` once and read the last line of the error.
Then put `#` in front of the line, so the notebook can run from top to bottom again.

In [ ]:
# your code here


In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert len(rows_1980) == 0
print("OK")

## 9.5 The main arms supplier

#### What you need

`sipri_suppliers` has one row per country, year and supplier, with the top 3 suppliers by name and
everyone else added up as `"Other"`. To find the biggest supplier:
1. filter to the country and year (two conditions, as in 9.1),
2. drop the `"Other"` row with `!=`,
3. sort by `tiv` from high to low with `sort_values("tiv", ascending=False)`,
4. take the first row with `.iloc[0]`.

**Why drop "Other":** it is the sum of many small suppliers, not one country. In Ukraine 2022, "Other" is
1,080 TIV and the United States 879. Without step 2 the card would say "Main arms supplier: Other",
which tells the reader nothing.

In [ ]:
# Example: run this cell
fruit = pd.DataFrame({"shop": ["A", "A", "A"], "kind": ["Apple", "Pear", "Other"], "kg": [30, 50, 90]})
fruit = fruit[fruit["kind"] != "Other"]
top = fruit.sort_values("kg", ascending=False).iloc[0]
print(top["kind"])

#### Your turn

**Step 1.** Write a function `main_supplier(iso3, year)` that returns the name of the biggest supplier,
or `"None"` if there is no supplier left after dropping "Other". Check `len(...) == 0` before `.iloc[0]`, as in 9.4.
Test it on Ukraine 2022, Ukraine 2014 and India 2000.

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

The four steps above are four lines. Put `if len(sup) == 0: return "None"` between step 2 and step 3.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert main_supplier("UKR", 2022) == "United States", "did you drop the Other row before sorting?"
assert main_supplier("UKR", 2014) == "None", "Ukraine 2014 only has an Other row"
assert main_supplier("IND", 2000) == "Russia"
assert main_supplier("CHE", 2000) == "None"
print("OK")

## 9.6 Put it together

**Step 1.** Write `year_card_values(iso3, year)`. It returns a list of four values in the order of the boxes:
`[regime, democracy index, conflict text, main supplier]`.
If there is no panel row, the first three are `"No data"`, `"n/a"`, `"n/a"`. The supplier is looked up either way.
Use the functions you already wrote.

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Filter, then `if len(rows) == 0:` with the three "no data" values, `else:` with the three real values
(using `rows.iloc[0]` and `conflict_text`). After the if/else, add `main_supplier(iso3, year)` as the fourth value.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert year_card_values("UKR", 2014) == ["electoral autocracy", 0.42, "4 · war", "None"]
assert year_card_values("UKR", 1980) == ["No data", "n/a", "n/a", "None"]
assert year_card_values("UKR", 2022) == ["electoral autocracy", 0.39, "2 · war", "United States"]
assert year_card_values("COL", 2000) == ["electoral democracy", 0.55, "1 · war", "United States"]
print("OK")

## Move it to the app: Task 9

Open `app/build_nicolas.py` and go to **TASK 9: the year card**. Delete the PLACEHOLDER line in `year_card`,
uncomment parts 1 to 4 (select the lines, Cmd + /) and fill the blanks. Everything you need is in 9.1 to 9.6.
The only new part is how the values are shown:

| In this notebook | In the app |
| --- | --- |
| `return [...]` with four values | `c1, c2, c3, c4 = st.columns(4)` and one `.metric` per box |
| `values = ["No data", "n/a", "n/a"]` | `c1.metric("Regime in " + str(year), "No data")`, `c2.metric("Democracy index", "n/a")`, ... |
| `round(row["democracy_index"], 2)` | `c2.metric("Democracy index", round(row["democracy_index"], 2))` |
| `return str(row["n_conflicts"]) + " · war"` | `c3.metric("Armed conflicts", str(row["n_conflicts"]) + " · war")` |
| `return top["supplier"]` | `c4.metric("Main arms supplier", top["supplier"])` |

**You will not see the year card yet.** The card is drawn inside `show_country`, which is still a placeholder
until task 10. That is expected: `check_build.py` will show Task 9 as OK once task 10 calls `year_card`.
Save the file and check that the browser shows no red box (no Python error). Then commit.

---
# Task 10: The country card with timelines

**What you will build:** a box with the country name, its code, the year card, four timelines under each
other (regime, democracy, conflicts, arms imports) and a fold-out table.

**Why:** this is the GeoPolHist idea from your project plan. Four thin charts with one shared year axis
read like one picture: you look straight down from a year and see the regime, the democracy score, the
conflicts and the arms imports of that year. The dashed line at the slider year helps the eye do that.

**How:** `show_country(name, year, p0, p1)` is a function, so the card is written once. Task 11 then calls it
twice for the comparison.

## 10.1 From name to code

#### What you need

The dropdown shows names, because people search for "Colombia", not "COL". All the tables are keyed by
`iso3`, because codes are the same in every dataset and names are not (you saw this in the crosswalk).
So the first line of the card turns the name into the code: filter by name, take the `iso3` column, take the first value.

#### Your turn

**Step 1.** Store the code of `"Colombia"` in `iso3`. Then store all years of that country in `rows`.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert iso3 == "COL"
assert len(rows) == 76, "all years 1950 to 2025 of Colombia"
print("OK")

## 10.2 Four charts with one year axis

#### What you need

`plt.subplots(4, 1, sharex=True)` makes 4 drawing areas under each other. `axes` is a list:
`axes[0]` is the top one, `axes[3]` the bottom one.
- `sharex=True`: all four share one x-axis. If you set the years on one of them, all follow.
- `gridspec_kw={"height_ratios": [1, 2, 1, 2]}`: the band charts (rows 0 and 2) are half as tall as the
  line and bar charts (rows 1 and 3). Bands only need a thin strip, numbers need room.
- `regime_bands(ax, spans, iso3, year)` and `conflict_bars(...)` are your functions from 07. They draw on
  the `ax` you give them, so you choose the row by passing `axes[0]` or `axes[2]`.

In [ ]:
# Example: run this cell
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 3), gridspec_kw={"height_ratios": [1, 2]})
axes[0].plot([2000, 2001, 2002], [9.1, 9.4, 9.0])
axes[1].bar([2000, 2001, 2002], [80, 120, 95])
axes[0].set_xlim(1999, 2003)       # sets the years for both rows
plt.tight_layout()
plt.show()

#### Your turn

**Step 1.** Draw the four timelines for Colombia with `year = 2000`, `p0 = 1950`, `p1 = 2025`
(use `iso3` and `rows` from 10.1):
- row 0: `regime_bands(axes[0], vdem_regime_spans, iso3, year)`
- row 1: `democracy_index` over `year` with `ax.plot`, y from 0 to 1, title "Democracy index", dashed line at `year`
- row 2: `conflict_bars(...)`
- row 3: `arms_imports_tiv` with `ax.bar`, title "Arms imports (TIV, millions)", dashed line at `year`

Then `axes[0].set_xlim(p0, p1)`, `plt.tight_layout()` and `plt.show()`. Keep the figure in `fig`.

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

The lines are already in the TASK 10 block of `build_nicolas.py`, part 2. Copy them here, fill the
blanks, and leave out `st.` lines for now. Use `plt.show()` in place of `st.pyplot(fig)`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert len(fig.axes) == 4, "plt.subplots(4, 1, ...)"
assert fig.axes[0].get_xlim() == (1950, 2025)
assert fig.axes[3].get_xlim() == (1950, 2025), "with sharex=True all rows share the years"
assert fig.axes[1].get_title(loc="left") == "Democracy index"
print("OK")

**Step 2.** Run the cell above again with `p0 = 1992`. What happens to all four rows? Write it in a comment here.

In [ ]:
# your code here


## 10.3 The table for the period

**Step 1.** The fold-out table under the chart shows only the years of the chosen period.
Make `table`: the rows of Colombia from 1992 to 2025 (two conditions with `>=` and `<=`).

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert len(table) == 34
assert table["year"].min() == 1992 and table["year"].max() == 2025
print("OK")

## 10.4 Put it together

**Step 1.** Write `country_figure(name, year, p0, p1)`. It finds the code and rows (10.1), draws the four rows (10.2),
zooms to the period and **returns** `fig` (no `plt.show()` inside). Test it with Iceland: no conflicts and
almost no arms imports. Does the conflict row show its "No conflict on record" message?

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
f = country_figure("Ukraine", 2014, 1992, 2025)
assert len(f.axes) == 4
assert f.axes[0].get_xlim() == (1992, 2025), "zoom with axes[0].set_xlim(p0, p1)"
plt.close(f)
f = country_figure("Iceland", 2000, 1950, 2025)
assert len(f.axes) == 4
plt.close(f)
print("OK")

## Move it to the app: Task 10

Go to **TASK 10** in `build_nicolas.py`. Delete the PLACEHOLDER line in `show_country` and uncomment parts 1 to 3.
Your `country_figure` is the middle of `show_country`. The new parts are the box around it and the lines that show things:

| In this notebook | In the app |
| --- | --- |
| (nothing) | `with st.container(border=True):` and everything below indented 4 more spaces |
| (nothing) | `st.subheader(name)` and `st.caption("ISO3 " + iso3)` at the top of the box |
| (nothing) | `year_card(iso3, year)`: your task 9 card, now finally drawn |
| `return fig` / `plt.show()` | `st.pyplot(fig)` then `plt.close(fig)` |
| `table = rows[...]` in 10.3 | inside `with st.expander("Year-by-year table for " + name):`, then `st.dataframe(table, hide_index=True)` |

Then in terminal 2: `python tools/check_build.py`. Tasks 9 and 10 should both say OK.
Look at Ukraine in the browser and drag nothing yet: the slider comes in task 11. Commit and push.

---
# Task 11: Period buttons, year slider, comparison

**What you will build:** three buttons (1950–2025 | Cold War | Since 1992), a year slider that stays inside the
chosen period, and a "Compare with" dropdown that puts a second country card next to the first.

**Why:** the period buttons let you tell a story in the presentation ("look at the Cold War only"). The
comparison is the most useful view for your question: a high importer next to a low importer, same years,
same scales. It is also the bonus feature the build guide recommends (see After task 12).

## 11.1 A dictionary from button text to years

#### What you need

The period button gives back its text, for example `"Cold War"`. A dictionary turns that text into the two
years in one lookup, with no `if / elif` chain. The value is a pair `(first, last)`, and
`p0, p1 = pair` takes the pair apart into two variables.

In [ ]:
# Example: run this cell
sizes = {"small": (2, 3), "large": (5, 8)}
lo, hi = sizes["large"]
print(lo, hi)

#### Your turn

**Step 1.** Make `periods` with the three periods: 1950 to 2025, Cold War 1950 to 1991, Since 1992 1992 to 2025.
The first key is `"1950–2025"` with a long dash: copy it from this sentence, do not type it.
Then unpack the Cold War years into `p0, p1`.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert list(periods.keys()) == ["1950–2025", "Cold War", "Since 1992"], "check the long dash in the first key"
assert (p0, p1) == (1950, 1991)
assert periods["Since 1992"] == (1992, 2025)
print("OK")

## 11.2 A start value that fits the period

#### What you need

`st.slider("Year", p0, p1, start)` stops with an error if `start` is outside `p0` to `p1`. 2000 is a good start,
but it is outside the Cold War. `min(max(2000, p0), p1)` keeps 2000 if it fits, and otherwise moves it to the
nearest edge of the period.

#### Your turn

**Step 1.** Write `start_year(p0, p1)` with that one line and test it on the three periods.
Then explain in a comment what `max` does here and what `min` does.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert start_year(1950, 2025) == 2000
assert start_year(1950, 1991) == 1991, "the Cold War ends before 2000"
assert start_year(1992, 2025) == 2000
assert start_year(2010, 2025) == 2010
print("OK")

## 11.3 The comparison list

#### What you need

A dropdown always has a value, and the first option is the start value. If the first option were a country,
the app would always show two cards. So the list starts with `"(none)"`, and the code only draws a second card
when the choice is something else. `["(none)"] + names` joins two lists into one.

**Step 1.** Make `options` and print its first three entries and its length.

In [ ]:
# your code here


**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert options[0] == "(none)"
assert len(options) == len(names) + 1
print("OK")

## 11.4 Two cards side by side (read, no code)

```python
if other == "(none)":
    show_country(country, year, p0, p1)
else:
    left, right = st.columns(2)
    with left:
        show_country(country, year, p0, p1)
    with right:
        show_country(other, year, p0, p1)
```

This is why `show_country` is a function: the whole card is written once and used twice. Both cards get the same
`year`, `p0` and `p1`, so the dashed lines and the zoom match, and you can compare the two countries row by row.

## Move it to the app: Task 11

Go to **TASK 11**. Delete the three PLACEHOLDER lines and uncomment the YOUR TURN block. Fill the blanks with
11.1 to 11.3 (`periods`, `periods[period]`, the slider with `p0`, `p1` and the start value, `["(none)"] + names`)
and 11.4 (`country` on the left, `other` on the right).
Run `python tools/check_build.py`: Task 11 OK. In the browser, try Cold War, then compare Ukraine with Poland.

---
# Task 12: The Data and method page

**What you will build:** three boxes with bullet points: Pipeline, Main finding, Caveats.

**Why:** in the Q&A someone will ask how the numbers were made. This page answers it inside the app, and it is
the same content as the "data and cleaning" and "analysis" parts of your report. Write the bullets in your own
words (the report has to be written without AI, so these are good practice).

**How:** first compute the facts here, so every number on the page is correct. Then write the text in the app.

**Step 1.** Compute and print:
- `n_countries`: the number of countries in `panel`
- `first_year`, `last_year`: the first and last year in `panel`
- `n_onsets`: the number of conflict onsets in `event` (one onset = one unique pair of `iso3` and `onset_year`)
- `high_10`, `low_10`: `dem_change` at `t = 10` for both groups in `event_time`

In [ ]:
# your code here


<details><summary><b>Hint 1</b> (open only if stuck)</summary>

`.nunique()` counts different values in a column. For the onsets, keep the two columns
`event[["iso3", "onset_year"]]` and use `.drop_duplicates()`, then `len(...)`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the step is done. Do not change it.

In [ ]:
assert n_countries == 182
assert (first_year, last_year) == (1950, 2025)
assert n_onsets == 281
assert round(high_10, 3) == 0.034 and round(low_10, 3) == 0.019
print("OK")

**Step 2.** Plan your bullets in this markdown cell (double-click to edit). Then write them into the app.

*Pipeline (3 to 5 bullets):* the three sources and versions; how they are matched (crosswalk to `iso3`);
one row per country and year in `panel.csv`; what an onset is and the window from 5 years before to 10 years after.

*Main finding (1 to 2 bullets):* what the chart shows, with the numbers from Step 1.

*Caveats (2 to 4 bullets):* ideas to think about:
- Does the chart show that arms imports cause anything?
- How are "high" and "low" importers split? (06, Part B, Step 4: the median of imports in the 5 years before the onset)
- What does TIV measure, and what does it not measure?
- Why do some countries have no data in some years (Ukraine 1980)?
- How many onsets are left at `t = 10` (`n_countries` in `event_time`)?

## Move it to the app: Task 12

Go to **TASK 12**, delete the PLACEHOLDER line and uncomment the block. In `st.markdown(""" ... """)`
every line that starts with `- ` becomes a bullet. Keep the bullets short: one line each.
Run `python tools/check_build.py`: all 12 tasks OK.

---
# After task 12

When `check_build.py` says "All 12 tasks pass", commit and push. Then, in this order:

1. **One colour system.** The map uses the colours in `regime_colours` at the top of `build_nicolas.py`
   (dark red for closed autocracy, blue for liberal democracy). The regime bands on the country card come
   from `panels_conflict.py`, which still has the colours from 07 (blue, orange, green, red). The same regime
   has two different colours on two pages. Copy the dictionary from the app into `panels_conflict.py` (and into
   07 so the notebook matches), then look at both pages.
2. **`plotly` in `requirements.txt`.** The app imports `plotly.express`, but `requirements.txt` does not list it.
   On Streamlit Cloud the app would stop at the import. Add a line `plotly`.
3. **Yair's charts.** Rows 1 and 3 of the country card (democracy line and arms bars) and the main finding chart
   are drawn directly in the app. When Yair has `democracy_line`, `arms_line` and `main_chart` in
   `panels_analysis.py` and `panels_arms.py`, replace those lines with his functions, as you did with
   `regime_bands` and `conflict_bars`. Agree with him who changes the app file, so you do not edit it at the same time.
4. **The final app.** Agree with Yair that this becomes the app: copy `build_nicolas.py` into `streamlit_app.py`,
   then deploy on Streamlit Cloud (share.streamlit.io, main file `app/streamlit_app.py`).
5. **Bonus feature.** The "Compare with" view from task 11 is the comparison the build guide recommends as the
   bonus. Confirm with Justinas that it counts, and describe it in the README in two sentences.
6. **README findings.** Your Main finding bullets from task 12 are a good base for the three findings the README still asks for.